# Route XML Filter
Objective: Filter the complete Bus Routes of Berlin accoarding to a csv containing the desired routes \
Author: Sven Vorgheim \
Disclaimer: The creation of this file was aided by ChatGPT

In [ ]:
# Imports
import pandas as pd
from lxml import etree
import math
import os

Assumes the use of "berlin_bus_rou.xml" as given in the BeST-Scenario \
Requieres a "depot_line_type.csv" containing a coloumn "Line" with the desired Lines as string

In [ ]:
# Read the csv, select coloumn
specified_routes = pd.read_csv("./depot_line_type.csv",sep=",")
specified = set(specified_routes["line"])
# Parse xml file
tree = etree.parse("../../sumo/berlin_bus.rou.xml")
root = tree.getroot()
# xml anchor contains line as the first part of the anchor
# removes any undesired routes and flows
for route in root.xpath("./route"):
    anchor = route.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(route)

for flow in root.xpath("./flow"):
    anchor = flow.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(flow)

# write result to file as intermediated chache
tree.write(
    "filtered_flows_routes.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)

Calculate additonal information from routes and flows \
Number of simultaneous busses is calculated from route duration and flow period \
nr_of_buses = route.duration/flow.period \
nr_of_repetitions = (flow_end-flow_begin)/duration \
Each Bus on the line must depart $n*period$ after n =1 

Define deadheading routes by Adding departure and arrival areas? Sumo Taz

In [ ]:
# Helper Function time in seconds from midnight
def parse_time(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

In [ ]:
tree = etree.parse("./filtered_flows_routes.xml")
root = tree.getroot()

rows = []

for route in root.xpath("./route"):
    anchor = route.get("id")
    for flow in root.xpath("./flow"):
        if flow.get("route") == anchor:
            period = float(flow.get("period"))
            duration = float(route.findall("stop")[-1].get("until"))
            nr_of_buses = math.ceil(int(duration) / int(period))
            flow_end = parse_time(flow.get("end"))
            flow_begin = parse_time(flow.get("begin"))
            # print(flow.get("begin"),flow_begin,flow.get("end"),flow_end)
            nr_of_repetitions = (flow_end-flow_begin)/duration
            # Add the repeat attribute to the route
            # route.set("repeat", str(int(nr_of_repetitions)))
            rows.append({
                "route": anchor,
                "flow_end": flow_end,
                "flow_begin": flow_begin,
                "period": period,
                "duration": duration,
                "nr_of_buses": nr_of_buses,
                "nr_of_repetitions": int(nr_of_repetitions),
            })
            root.remove(flow)

# write final result to file
tree.write(
    "cicero_mueller_routes.rou.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)
os.remove("./filtered_flows_routes.xml")

route_calculations = pd.DataFrame(rows)
print(route_calculations)
# Total result of simultaneous buses
print(sum(route_calculations["nr_of_buses"]))
print(sum(route_calculations["nr_of_buses"]*route_calculations["nr_of_repetitions"]))
route_calculations.to_csv("route_calculations.csv", index=False)

Created merged csv containing information on Lines

In [ ]:
route_df = pd.read_csv("./route_calculations.csv")
line_df = pd.read_csv("./depot_line_type.csv")

# Extract the line name from the route ID
route_df["line"] = route_df["route"].str.split("_").str[0]

# Join on the Line column
merged = route_df.merge(line_df, on="line", how="left")

merged.to_csv("merged_routes.csv", index=False)
os.remove("./route_calculations.csv")
print(merged.head())

In [ ]:

# Read the csv, select coloumn
specified_routes = pd.read_csv("./depot_line_type.csv",sep=",")
# Create a lookup indexed by route
print(specified_routes.head())
route_lookup = specified_routes.set_index("route")[["depot", "type"]]

# Parse XML
cm_tree = etree.parse("./cicero_mueller_routes.rou.xml")
cm_root = cm_tree.getroot()

for route in cm_root.xpath("./route"):
    route_id = route.get("id")

    if route_id in route_lookup.index:
        if route_lookup.at[route_id, "depot"] == "Müllerstraße":
            depot_taz = "taz_Depot_Muellerstrasse"
        if route_lookup.at[route_id, "depot"] == "Cicerostraße":
            depot_taz = "taz_Depot_Cicerostrasse"
        route_type = route_lookup.at[route_id, "type"]
        print(route_id, depot, route_type)

# write final result to file
tree.write(
    "cicero_mueller_deadheads.rou.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)

route_calculations = pd.DataFrame(rows)
print(route_calculations)
# Total result of simultaneous buses
print(sum(route_calculations["nr_of_buses"]))
route_calculations.to_csv("route_calculations.csv", index=False)

In [ ]:
# Store unique final station IDs
final_station_ids = set()

# Parse XML files
fs_tree = etree.parse("../route_building/cicero_mueller_routes.rou.xml")
fs_root = fs_tree.getroot()

bus_stops_tree = etree.parse("../../sumo/berlin_bus_stops.add.xml")
bus_stops_root = bus_stops_tree.getroot()

# Collect unique final station IDs
for route in fs_root.findall("route"):
    stops = route.findall("stop")
    if stops:
        final_station_ids.add(stops[-1].get("busStop"))

# Collect information about those stations
station_values = []

for bus_stop in bus_stops_root.findall("busStop"):
    bus_stop_id = bus_stop.get("id")

    if bus_stop_id in final_station_ids:
        station_values.append({
            "id": bus_stop_id,
            "lane": bus_stop.get("lane"),
            "startPos": bus_stop.get("startPos"),
            "endPos": bus_stop.get("endPos"),
            "name": bus_stop.get("name"),
            "friendlyPos": bus_stop.get("friendlyPos"),
            "parkingLength": bus_stop.get("parkingLength"),
        })

